# 03a · SVM

**Owner:** Pranav

Tunes linear and RBF SVMs as one model family (at most 20 configurations), selects the best configuration with stratified 5-fold CV on the training set, analyses it on out-of-fold (OOF) predictions, and saves the OOF scores for threshold selection in `04_evaluation`.

Shared repo code: `src.data` (loading), `src.preprocessing` (features + pipeline), `src.validation` (folds, OOF scores, F2 threshold, metrics). No test data is used in this notebook.

**Outputs**
- `results/tuning/svm_configs.csv`: every configuration with all CV metrics
- `results/scores/svm_oof.csv`: OOF scores of the best SVM (`PatientID, y_true, score`)
- `results/models/svm.joblib`: best pipeline refitted on the full training set
- `results/figures/models/svm_*.png`: diagnostic figures

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, learning_curve
from sklearn.metrics import precision_recall_curve, average_precision_score, ConfusionMatrixDisplay

from src.data import load_train
from src.preprocessing import add_engineered_features, build_preprocessor
from src.validation import get_cv, oof_scores, save_oof, best_f2_threshold, classification_metrics

RESULTS = os.path.join("..", "results")
FIG_PATH = os.path.join(RESULTS, "figures", "models")
for sub in ["tuning", "models", "scores", os.path.join("figures", "models")]:
    os.makedirs(os.path.join(RESULTS, sub), exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, name), dpi=150, bbox_inches="tight")
    plt.show()

## 1. Data and features
Training set only, all features (`load_train()` + `add_engineered_features()`). The preprocessing (`build_preprocessor(use_engineered=True, use_bins=True)`, see notebook 02) scales the continuous features and the two symptom counts, one-hot encodes Ethnicity, passes the binary features and EducationLevel through, and adds 10 quantile bins (one-hot) for UPDRS, MoCA and FunctionalAssessment to capture their threshold-like relationship with Diagnosis. The test set stays locked (`load_test()` refuses) until thresholds are frozen in `04_evaluation`.

In [ ]:
X_train, y_train = load_train()
X_train = add_engineered_features(X_train)
print(X_train.shape, "| PD share:", round(y_train.mean(), 3))

## 2. Baseline
With 62% PD, predicting *everyone* as PD already reaches a high F2 (precision = 0.62, recall = 1) but has specificity 0. Every model is judged against this.

In [ ]:
p = y_train.mean()
baseline = {"model": "baseline (all PD)", "TP": int(y_train.sum()), "FN": 0,
            "FP": int((1 - y_train).sum()), "TN": 0, "recall": 1.0, "precision": p,
            "specificity": 0.0, "F2": 5 * p / (4 * p + 1)}
pd.DataFrame([baseline]).round(3)

## 3. Validation strategy
- **Held-out test set (20%, stratified)** was created once in notebook 01 and is never used here.
- **Stratified 5-fold CV** on the 1,684 training patients (`get_cv()`, seed 42): each fold keeps the 62/38 class ratio and has ~337 validation patients, and SVM, MLP and FIS all use the *same* folds, so their scores are directly comparable. Five folds balance the variance of the estimate against compute time while keeping 80% of the data for training in each fold.
- **No leakage:** scaling, one-hot encoding and the quantile bins sit inside the `Pipeline`, so they are fitted on the training folds only (the bin edges are learned per fold and never use the labels).

## 4. Hyperparameter search (17 of max 20 configurations)

| Kernel | C | gamma | Configs |
|---|---|---|---|
| linear | 0.01, 0.1, 1, 10, 100 | – | 5 |
| rbf | 0.1, 1, 10, 100 | scale, 0.01, 0.1 | 12 |

- **C** controls the margin/regularisation trade-off: small C = wider margin, more regularisation.
- **gamma** controls how local the RBF kernel is. `scale` = 1 / (n_features · Var(X)), the scikit-learn default; 0.01 and 0.1 are smoother / more local alternatives around it.
- Exponentially spaced C and gamma follow the grid-search recommendation of Hsu, Chang & Lin (2003), *A Practical Guide to Support Vector Classification*.
- Both kernels are candidates **within one SVM family**: the single best configuration goes forward to the comparison with MLP and FIS.

**Selection metric: average precision (AP).** The decision threshold (max F2) is chosen *after* tuning, so the tuning metric should judge how well a configuration **ranks** PD patients above non-PD patients, independent of any threshold. AP (area under the precision–recall curve) is threshold-free and focuses on the PD class, matching the recall-oriented goal. F2 or recall at the default threshold would judge a cut-off that is replaced later and can favour configurations that predict PD for almost everyone.

In [ ]:
pipe = Pipeline([
    ("prep", build_preprocessor(use_engineered=True, use_bins=True)),
    ("svc", SVC()),
])
param_grid = [
    {"svc__kernel": ["linear"], "svc__C": [0.01, 0.1, 1, 10, 100]},
    {"svc__kernel": ["rbf"], "svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1]},
]

search = GridSearchCV(pipe, param_grid, cv=get_cv(), scoring="average_precision",
                      return_train_score=True, n_jobs=-1)
search.fit(X_train, y_train)
n_configs = len(search.cv_results_["params"])
assert n_configs <= 20, "Assignment limit: at most 20 SVM configurations"
print("Configurations evaluated:", n_configs)

## 5. Results of all configurations
Each configuration is trained on 4 folds and validated on the 5th, five times; the table shows the mean (± std) validation AP, the mean train AP, and the gap between them (larger gap = more overfitting). Sorted by validation AP.

In [ ]:
res = pd.DataFrame(search.cv_results_)
configs = pd.DataFrame({
    "kernel": res["param_svc__kernel"],
    "C": res["param_svc__C"].astype(float),
    "gamma": res["param_svc__gamma"].fillna("-").astype(str),
})
configs["val_AP"] = res["mean_test_score"].round(4)
configs["val_AP_std"] = res["std_test_score"].round(4)
configs["train_AP"] = res["mean_train_score"].round(4)
configs["overfit_gap"] = (configs["train_AP"] - configs["val_AP"]).round(4)
configs["rank"] = res["rank_test_score"]
configs = configs.sort_values("rank")
configs.to_csv(os.path.join(RESULTS, "tuning", "svm_configs.csv"), index=False)
configs

In [ ]:
# Best configuration per kernel: the two candidates within the SVM family
configs.groupby("kernel").head(1)

**Selected SVM:** the configuration with the highest mean validation AP across both kernels (linear SVM).


In [ ]:
best_params = search.best_params_
best = configs.iloc[0]
print(f"BEST SVM: kernel={best.kernel}, C={best.C:g}, gamma={best.gamma} "
      f"| validation AP = {best.val_AP:.3f} ± {best.val_AP_std:.3f}")

### 5.1 RBF hyperparameter landscape
Validation AP for every C × gamma combination of the RBF kernel: shows the best region and where larger C or gamma start to overfit. (The linear kernel only has C; its results are in the table above.)

In [ ]:
rbf = configs[configs.kernel == "rbf"].pivot(index="gamma", columns="C", values="val_AP")
plt.figure(figsize=(6.5, 4))
sns.heatmap(rbf, annot=True, fmt=".3f", cmap="viridis")
plt.title("RBF: validation AP by C and gamma")
savefig("svm_rbf_heatmap.png")

## 6. Learning curve of the best configuration (Linear SVM)
How validation AP changes with the amount of training data: a large, persistent train–validation gap points to variance (more data would help); two low, converged curves point to bias.

In [ ]:
sizes, tr, va = learning_curve(
    pipe.set_params(**best_params), X_train, y_train,
    train_sizes=[0.1, 0.25, 0.5, 0.75, 1.0], cv=get_cv(),
    scoring="average_precision", n_jobs=-1)

plt.figure(figsize=(6, 4))
plt.plot(sizes, tr.mean(axis=1), marker="o", label="Train")
plt.plot(sizes, va.mean(axis=1), marker="o", label="Validation (CV)")
plt.fill_between(sizes, va.mean(axis=1) - va.std(axis=1), va.mean(axis=1) + va.std(axis=1), alpha=0.2)
plt.xlabel("Training patients per fold"); plt.ylabel("Average precision")
plt.title("SVM learning curve (best configuration)"); plt.legend()
savefig("svm_learning_curve.png")
pd.DataFrame({"train_size": sizes, "train_AP": tr.mean(axis=1).round(4), "val_AP": va.mean(axis=1).round(4)})

## 7. Linear vs RBF on out-of-fold predictions
**What the code does:** takes the best linear and the best RBF configuration from the table above (printed below) and runs each through `oof_scores()`. Every training patient then gets one score from a model that was trained on the other 4 folds (training data only; the test set is untouched).

- **Table:** each kernel's results when the F2 rule is applied to its OOF scores, next to the all-PD baseline. The threshold here is a *preview*; the official one is chosen and frozen in `04_evaluation`.
- **Precision–recall curve:** precision (y) against recall (x) for every possible threshold. A curve that stays higher ranks patients better; the area under it is the AP. The dotted line is the baseline precision (62% PD).

In [ ]:
cand = configs.groupby("kernel").head(1)
print(cand[["kernel", "C", "gamma", "val_AP"]].to_string(index=False))
oof = {}
for _, row in cand.iterrows():
    params = {"svc__kernel": row.kernel, "svc__C": row.C}
    if row.kernel == "rbf":
        params["svc__gamma"] = "scale" if row.gamma == "scale" else float(row.gamma)
    oof[row.kernel] = oof_scores(pipe.set_params(**params), X_train, y_train)

rows = [baseline]
for k, s in oof.items():
    t, _ = best_f2_threshold(y_train, s)
    r = classification_metrics(y_train, s, t, f"{k} (OOF, best-F2 thr)")
    r.update({"AP": average_precision_score(y_train, s), "threshold": t})
    rows.append(r)
oof_table = pd.DataFrame(rows).round(3)
oof_table

In [ ]:
plt.figure(figsize=(6, 4.5))
for k, s in oof.items():
    prec, rec, _ = precision_recall_curve(y_train, s)
    plt.plot(rec, prec, label=f"{k} (AP={average_precision_score(y_train, s):.3f})")
plt.axhline(y_train.mean(), color="grey", ls=":", label="Baseline (PD share)")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–recall, linear vs RBF (OOF)"); plt.legend()
savefig("svm_pr_curve.png")

## 8. Best SVM (selected in section 5): scores, threshold and confusion matrix
**What is the score and the threshold?** For every patient the SVM outputs a **decision score**: the signed distance to its decision boundary. Positive = PD side, negative = non-PD side, further from 0 = more confident. To turn scores into a yes/no prediction we pick a **threshold**: every patient with score ≥ threshold is predicted PD. The assignment's rule is to pick the threshold that **maximises F2**. Here that rule is applied to the OOF scores as a *preview*; the official threshold is chosen and frozen in `04_evaluation`.

The three panels below:
1. **Score distribution by class:** x = OOF decision score, y = number of patients; blue = actually PD, orange = actually non-PD. The dashed line is the best-F2 threshold: everyone to its right is predicted PD. Where the two colours overlap, no threshold can separate the classes perfectly.
2. **Metrics vs threshold:** x = a candidate threshold, y = the metric's value if that threshold were used. Moving the threshold right flags fewer patients as PD: recall falls, specificity and precision rise. F2 peaks at the dashed line.
3. **Confusion matrix at the best-F2 threshold:** rows = true class, columns = predicted class.

In [ ]:
best_kernel = search.best_params_["svc__kernel"]
svm_oof = oof[best_kernel]
t_best, f2_best = best_f2_threshold(y_train, svm_oof)

ths = np.quantile(svm_oof, np.linspace(0.01, 0.99, 200))
sweep = pd.DataFrame([classification_metrics(y_train, svm_oof, t) for t in ths]).assign(threshold=ths)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for cls, lab in [(1, "PD"), (0, "Non-PD")]:
    axes[0].hist(svm_oof[y_train == cls], bins=40, alpha=0.6, label=lab)
axes[0].axvline(t_best, color="k", ls="--", label=f"best-F2 threshold ({t_best:.2f})")
axes[0].set_xlabel("SVM decision score (OOF)"); axes[0].set_title("Score distribution by class"); axes[0].legend()

for m in ["recall", "precision", "specificity", "F2"]:
    axes[1].plot(sweep.threshold, sweep[m], label=m)
axes[1].axvline(t_best, color="k", ls="--")
axes[1].set_xlabel("Threshold"); axes[1].set_title("Metrics vs threshold (OOF)"); axes[1].legend()

y_pred = (svm_oof >= t_best).astype(int)   # preview only; threshold frozen in 04
ConfusionMatrixDisplay.from_predictions(y_train, y_pred, display_labels=["Non-PD", "PD"],
                                        cmap="Blues", ax=axes[2], colorbar=False)
axes[2].set_title(f"Confusion matrix at best-F2 threshold (OOF)")
savefig("svm_threshold_analysis.png")

print(f"Best-F2 threshold (preview) = {t_best:.2f}  ->  F2 = {f2_best:.3f}")

## 9. Save OOF scores and the final model
`save_oof()` writes the best SVM's OOF scores in the agreed format for `04_evaluation`. The model is the whole pipeline (preprocessing + SVC) refitted on the full training set (`refit="AP"`), so notebook 04 only needs `add_engineered_features(X_test)` before `decision_function`. SVM scores are decision-function values (not probabilities), so the SVM threshold is on that scale.

In [ ]:
print("Saved:", save_oof("svm", y_train, svm_oof))
joblib.dump(search.best_estimator_, os.path.join(RESULTS, "models", "svm.joblib"))
print("Saved results/models/svm.joblib")

## 10. Notes for the report
- Linear vs RBF (is the gap larger than the fold std? OOF curves):
- Selection metric: why AP (threshold-free ranking, PD-focused):
- Over/underfitting (train vs validation AP, learning curve):
- OOF F2 vs baseline F2; specificity and the threshold trade-off:
- Limitation: the same folds were used to tune and to produce OOF scores, so OOF metrics are slightly optimistic; the test set gives the unbiased estimate.